## F1 RACE FINISHING POSITION PREDICTOR

### Goal

We want to predict the finishing position of race drivers in a Formula 1 grand prix, based on historical performances of the drivers/ teams, characteristics of the tracks, and race specific (FP performance, qualifying, weather etc) and season specific (team and driver momentum) details. 

#### Challenges

This problem is hard because we need a model that is : 
- Robust to outliers (DNFs, safety cars, weather, team strategies, etc)
- Deals well with tabular, mixed type features.
- Captures Non linear interactions (fast track + slippery conditions + low overtaking ability)

#### **XGBoost**
We choose XGboost over other models because, linear models cannot fit non linear interactions, random forests are a good baseline, but still lag behind Boosting as boosting predicts residuals with new instead of averaging down new trees. Neural networks need a lot more data, gradient boosting doesn't internally provide regularization (can over fit to outliers easily) . Other boosting models like LightGBM and CatBoost are good alternatives, and is the planned second iteration. [An easy benefit of XGboost is the native handling of missing values]. 

#### Success Metrics

We evaluate the model on the held out set of recent races using:

  - MAE on finishing position 
        Target: < 3.0 positions on average
  
  - Top-3 (podium) hit rate
        Target : > 60% of actual podium finishers in our predicted top 3. 
  
  - Top-10 (points) hit rate 
        Target: > 75% 

  - Spearman rank correlation per race
        Target: > 0.65 averaged across test races.
        Rationale: To represent models ability to get the order correct.

#### Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

import xgboost as xgb

We do not use Sickit learn's `GradientBoostingRegressor` because it is materially slower. 

### Configuration

We centralize the configuration - meaning hyperparameter settings can all be tracked and set in one place. 

In [ ]:
CONFIG = {
    # path to data file
    "data_path": "data/f1_model_data.parquet",

    # training seasons
    "train_seasons": list(range(2014, 2024)),
    # test seasons
    "test_seasons": [2024],

    # --Hyperparameters for XGBoost--

    "xgb_params": {

        # Loss function for residuals. Matches our MAE evaluation metric. 
        "objective": "reg:squarederror",

        # Number of trees in the ensemble. We Set-high, and rely on early-stopping to pick the best iteration
        "n_estimators":2000,

        # Max depth each tree should grow. F1 maybe has ~30-50 useful features, depth 6 lets the model
        # learn interactions without memorizing. Depth >8 on this size overfits quickly. 
        "max_depth": 6,

        # Learning rate - small LR + many trees is standard. Too big underfits the residual xgb structure.
        "learning_rate": 0.05,

        # Row subsampling helps in regularization. Each tree sees 80% of rows which decorrelates trees and reduces overfitting..
        "subsample": 0.8,

        # column subsampling helps in regularization, reduces overfitting to certain features. Each tree sees 80% of columns.
        "colsample_bytree": 0.8,

        # L2 regularization term on weights. Discourages any single leaf from dominating. 
        "reg_lambda": 1.0, 

        # Min. Child weight - minimum sum of hessians (in our case, hessians for all rows is 1, so it implies the min number of rows in the leaf)
        "min_child_weight": 1,

        # without this every run would be different.Allows reproducibility of results.
        "random_state": 42,   

        # Typically 5-10x faster than exact algo without loss to accuracy.
        "tree_method": "hist", 

        # Suppress per-iteration warnings
        "verbosity": 1,

    }, 


# --- Cross-Validation ---

# We group by 'race_id' so a single race doesn't get split up between train/test folds. This is important to prevent data about the race from leaking into training.

"cv_folds":5,

# Early stopping rounds - stop if validation MAE doesn't improve for this many iterations. 

"early_stopping_rounds": 50,

}

### Data Loading

Simple function to lead the data parquet file into a dataframe. 

In [ ]:
def load_data(path:str) -> pd.DataFrame:
    df = pd.read_parquet(path)

    # Optional : Sanity checks - 
    assert df["Position"].between(1, 21).all()
    assert df["race_id"].notna().all()

    return df

##### **Split Features - Target**

In [ ]:
def split_features_target(df: pd.DataFrame):

    """
    Separate the dataframe into:
    X  — feature matrix
    y  — target vector (finishing_position)
    groups — race_id, used for group-aware CV
    """

    target_col = "Position"

    # We drop identifiers to risk overfitting to certain drivers/ tracks, and rely on
    # our features to capture the relevant information about the driver and the track.

    drop_cols = ["Position", "Season", "Round", "race_id",
             "Abbreviation", "TeamName", "Location", "Circuit"]

    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols].copy()
    y = df[target_col].copy()
    groups = df["race_id"].copy()

    # For now we do not have any categoricaL features, but if we decide to add them - 
    
    # Convert object/string columns to categorical for XGBoost native handling.
    for col in X.select_dtypes(include=["object"]).columns:
        X[col] = X[col].astype("category")

### Training   

In [ ]:
def train_with_cv(X: pd.DataFrame, y: pd.Series, groups: pd.Series, config: dict):

    """ 
    Train XGBoost using GroupKFold cross-validation

    Returns a list of trained models (one per fold). Ensembling fold models at inference time gives an MAE improvement.
    Final predictions are the mean of five GroupKFold models; averaging decorrelated fold models reduces variance.
    """

    cv = GroupKFold(n_splits=config["cv_folds"])
    fold_models = []
    fold_metrics = []

    # The for loop below emulates a cross validation style of training models - splitting rows into train and validation sets, 
    # training a model on the train set, and evaluating it on the validation set. 
    # -- Cross Validation --
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y, groups)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Create an instance of the model
        model = xgb.XGBRegressor(
        **config["xgb_params"], # unpacking the dict of params
        enable_categorical=True,
        early_stopping_rounds=config["early_stopping_rounds"],
        )

        # Train the model on the training fold
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )

        # predict on the eval set and collect relevant metrics for each fold.

        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        fold_metrics.append({"fold": fold_idx, "mae": mae,
                                "best_iter": model.best_iteration})
        fold_models.append(model)

        print(f"[fold {fold_idx}] MAE={mae:.3f}  best_iter={model.best_iteration}")

### Evaluation


We use the models to make predictions on the held out test data. The ensemble of models make a prediction - their mean is used as the final output. 
Then the metrics discussed being within the scope of the project, are calculated.

The model predicts each driver's expected finishing position, but what we actually consume is the order those predictions produce once sorted. 
- **Raw MAE** is reported for calibration, but understates the model — because it's trained on squared error, predictions compress toward the mid-field (a likely winner scores ~3.5, not 1.0), so MAE is penalized even when the induced order is correct.
- **Rank MAE** corrects for this by scoring the sorted positions instead of the raw values,
- **Spearman correlation** is the Pearson correlation computed on the ranks of two variables rather than their raw values, so it measures how well one ordering matches another on a scale from −1 (reversed) to +1 (identical), capturing any monotonic relationship regardless of magnitude, rewarding correct ordering independent of absolute values.
- **Podium and points hit rates** Give the insight that fans care about the most - if the model can predict podium and points well. 

In [ ]:
def evaluate(models, X_test: pd.DataFrame, y_test: pd.Series,
             race_ids: pd.Series) -> dict:

    # We take in the models, the test input features and target,
    # and race ids of the test years (to calc the metrics per race)
    """
    Evaluate the ensemble on held-out races.
    """

    preds = np.mean([m.predict(X_test) for m in models], axis=0)

    overall_mae = mean_absolute_error(y_test, preds)

    # Per-race metrics
    df_eval = pd.DataFrame({
        "race_id": race_ids.values,
        "y_true": y_test.values,
        "y_pred": preds,
    })

    spearman_per_race = []
    podium_hits = []
    points_hits = []
    rank_mae = []

    # Calculate per-race metrics 
    for _, race in df_eval.groupby("race_id"):

        # Spearman rank correlation between predicted and actual finishing
        # order. 
        rho, _ = spearmanr(race["y_true"], race["y_pred"])
        spearman_per_race.append(rho)

        # Hit rate = |predicted top-k ∩ actual top-k| / k
        actual_top3 = set(race.nsmallest(3, "y_true").index)
        pred_top3   = set(race.nsmallest(3, "y_pred").index)
        podium_hits.append(len(actual_top3 & pred_top3) / 3)

        actual_top10 = set(race.nsmallest(10, "y_true").index)
        pred_top10   = set(race.nsmallest(10, "y_pred").index)
        points_hits.append(len(actual_top10 & pred_top10) / 10)

        race["y_rank"] = race["y_pred"].rank(method="first")
        rank_mae_per_race = mean_absolute_error(race["y_true"].rank(method="first"), race["y_rank"])
        rank_mae.append(rank_mae_per_race)

    return {
        "mae": overall_mae,
        "rank_mae": np.mean(rank_mae),
        "spearman_mean": np.mean(spearman_per_race),
        "podium_hit_rate": np.mean(podium_hits),
        "points_hit_rate": np.mean(points_hits),
    }


### `Main()`

In [ ]:
def main():
    # 1. Load
    df = load_data(CONFIG["data_path"])

    # 2. Train/test split by season — see CONFIG comment on why not random.
    train_df = df[df["Season"].isin(CONFIG["train_seasons"])]
    test_df  = df[df["Season"].isin(CONFIG["test_seasons"])]

    X_train, y_train, groups_train = split_features_target(train_df)
    X_test,  y_test,  groups_test = split_features_target(test_df)

    # 3. Train with grouped CV
    models, fold_metrics = train_with_cv(X_train, y_train, groups_train, CONFIG)

    cv_mae = np.mean([m["mae"] for m in fold_metrics])
    print(f"\nCV MAE (mean across folds): {cv_mae:.3f}")

    # 4. Evaluate on held-out season
    results = evaluate(models, X_test, y_test, groups_test)
    
    print(f"\n==== Held-out Test Results for {CONFIG['test_seasons'][0]} - {CONFIG['test_seasons'][-1]} ====")
    print("\n=== Held-out Test Results ===")
    print(f"  MAE              : {results['mae']:.3f}   (target: < 3.0)")
    print(f"  Rank MAE         : {results['rank_mae']:.3f} (target: < 3.0)")
    print(f"  Spearman (mean)  : {results['spearman_mean']:.3f} (target: > 0.65)")
    print(f"  Podium hit rate  : {results['podium_hit_rate']:.3f} (target: > 0.60)")
    print(f"  Points hit rate  : {results['points_hit_rate']:.3f} (target: > 0.75)")

    print("Tested on seasons:", CONFIG["test_seasons"])


if __name__ == "__main__":
    main()